In [1]:
import torch
import torch.nn as nn

## PART 1: The CRNN Model Definition

In [2]:
class CRNN(nn.Module):
    def __init__(self, num_classes=10):
        super(CRNN, self).__init__()
        
        # 2. The CNN (Sound Feature Extractor)
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        
        self.block2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        
        # 4. The RNN (Rhythm Tracker)
        # Input size is 64 channels * 32 mel bins = 2048
        self.lstm = nn.LSTM(input_size=2048, hidden_size=64, num_layers=1, 
                            bidirectional=True, batch_first=True)
        
        # 5. The Final Prediction
        # Bidirectional LSTM concatenates the hidden states, so 64 * 2 = 128
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        # x shape initially: (Batch, 1, Mels, Time)
        x = self.block1(x)
        x = self.block2(x)
        
        # Store the shape for Q4 before reshaping
        self.q4_shape = x.shape
        
        # 3. The Bridge (Reshaping)
        # Flatten channels and frequencies: (Batch, Channels, Mels, Time) -> (Batch, Time, Channels * Mels)
        b, c, m, t = x.size()
        x = x.permute(0, 3, 1, 2).contiguous() # Move Time to dimension 1
        x = x.view(b, t, c * m)                # Flatten Channels and Mels
        
        # Pass through LSTM
        x, _ = self.lstm(x)
        
        # Global Max Pooling across the time dimension
        x, _ = torch.max(x, dim=1)
        
        # Final classification
        out = self.fc(x)
        return out

## PART 2: Calculations & Outputs

In [3]:
# Q1 Calculation
samples_per_genre = 50
genres = 10
q1_ans = samples_per_genre * genres
print(f"Q1) Total .wav files generated: {q1_ans}")

# Q2 Calculation
target_sr = 22050
duration = 30
total_samples = target_sr * duration
q2_ans = (1, total_samples) # Mono audio
print(f"Q2) Waveform tensor shape: {q2_ans}")

Q1) Total .wav files generated: 500
Q2) Waveform tensor shape: (1, 661500)


In [4]:
# Q3 Calculation (Mel-spectrogram shape)
# torchaudio calculates time frames as: floor(total_samples / hop_length) + 1
hop_length = 512
n_mels = 128
time_frames = (total_samples // hop_length) + 1
q3_ans = (1, n_mels, time_frames)
print(f"Q3) Pre-computed tensor shape: {q3_ans}")


# Q4 & Q5 Calculations (Model internals)
model = CRNN(num_classes=10)
dummy_input = torch.randn(32, 1, 128, 1292) # Batch of 32, using the shape from Q3
_ = model(dummy_input) # Run a forward pass to trigger internal shape tracking

q4_ans = tuple(model.q4_shape)
print(f"Q4) Shape immediately after second MaxPool2d: {q4_ans}")

q5_ans = sum(p.numel() for p in model.lstm.parameters() if p.requires_grad)
print(f"Q5) Trainable parameters in the LSTM layer: {q5_ans}")

print("\nQ6) Conceptually problematic Y-axis shift answer:")
print("> Moving a pattern up the Y-axis fundamentally changes its pitch and instrument identity (e.g., shifting a low bassline up 80 bins turns it into a high-pitched synth).")

Q3) Pre-computed tensor shape: (1, 128, 1292)
Q4) Shape immediately after second MaxPool2d: (32, 64, 32, 323)
Q5) Trainable parameters in the LSTM layer: 1082368

Q6) Conceptually problematic Y-axis shift answer:
> Moving a pattern up the Y-axis fundamentally changes its pitch and instrument identity (e.g., shifting a low bassline up 80 bins turns it into a high-pitched synth).
